# L-conjugacy lab

The workflow of the original L-conjugacy app (Polyscope, commit `9624f23`) as a notebook.
The logic lives in `Lconjugacy/pipeline.py`; this notebook sets the parameters,
runs each stage and shows the result in the kayviz viewer.

**How to use:** choose `SURFACE` in the first cell and run the cells in order. Three examples
come with their settings (range, sampling, radius, angle): `nautilus` and `seashell` (examples
of the paper) and `testSimpleOptimized`; their B-splines are in `data/bsplines/`. Stage 3
needs the external quad remesher (`st.path_remesher`); stages 0–2 run without it.

| Stage | What it does | Writes to `notebooks/out/lconjugacy/<surface>/<experiment>/` (git-ignored) |
|---|---|---|
| 0 | view the B-spline, choose the (u, v) range and sampling, optionally swap u ↔ v (flips the normals) | — |
| 0b | fit the spheres: radius function + dense reference for that range | — |
| 1 | optional B-spline optimisation (K > 0) | optional surface JSON |
| 2 | L-conjugate directions from the initial angles | — |
| 3 | cross fields → external quad remesher (`t` and `lc`) → load the remeshed quads | `mesh.obj`, `t_*/lc_*.dat`, `*_Remeshed.obj`, `pre_remesh_state.pkl` |
| 4 | L-mesh on the `t` quads (spheres on quad faces, planes on inner quad vertices) → post-optimisation → export | `lmesh*` export, `sphereCenter*.obj` |

In [ ]:
# Single-threaded BLAS: OpenBLAS's thread pool can deadlock in this kernel (the cell then
# hangs at 0% CPU inside numpy.linalg). Must run before numpy is first imported, i.e. at
# the top of a fresh kernel; the matrices here are tiny, so nothing gets slower.
import os, sys
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
if "numpy" in sys.modules:
    print("note: numpy was already imported - restart the kernel for single-threaded BLAS")

# Reload edited modules (e.g. LconjugacyGUI/pipeline.py) before each cell runs
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for p in (PROJECT_ROOT,):
    if p not in sys.path:
        sys.path.insert(0, p)

import kayviz as kv
from kayviz import serializers as ser
import Lconjugacy.pipeline as pl

kv.init()

# Example settings, taken from the original runs. Any file in data/bsplines/ (without .json)
# also works as SURFACE; it then starts from DEFAULT.
EXAMPLES = {
    # paper example (nautilus/ex2)
    "nautilus": dict(interval_u=[0.10, 0.90], interval_v=[0.30, 1.00], u_samples=50, v_samples=80,
                     radius_percentage=0.95, theta=45.0, flip_normals=False),
    # paper example (seashell/ex5)
    "seashell": dict(interval_u=[0.033, 0.84], interval_v=[0.29, 1.00], u_samples=40, v_samples=150,
                     radius_percentage=0.75, theta=45.0, flip_normals=False),
    "testSimpleOptimized": dict(interval_u=[0.10, 0.90], interval_v=[0.10, 0.90], u_samples=40, v_samples=40,
                                radius_percentage=0.75, theta=45.0, flip_normals=True),
}
DEFAULT = dict(interval_u=[0.2, 0.8], interval_v=[0.2, 0.8], u_samples=20, v_samples=20,
               radius_percentage=0.75, theta=30.0, flip_normals=False)

SURFACE    = "nautilus"              # "nautilus", "seashell", "testSimpleOptimized", or any data/bsplines/ file
EXPERIMENT = "experiment0"
EX = EXAMPLES.get(SURFACE, DEFAULT)

In [ ]:
# ── viewer helpers ────────────────────────────────────────────────────────────
from hanan import glyphs

CROSS_LEN = 0.02              # half-length of the drawn cross-field arms

def show_surface(st, clear=True):
    V, F, K, n = pl.surface_mesh(st)
    if clear:
        kv.clear()
    kv.register_surface_mesh("B-spline surface", V, F, color=(0.5, 0.5, 0.5), show_edges=True)
    kv.add_scalar_quantity("B-spline surface", "Gaussian curvature", K)
    P, E = pl.control_net(st)
    kv.register_curve_network("Control net", P, np.asarray(E), color=(1, 0, 0), radius=0.001)
    kv.register_point_cloud("Control points", P, color=(0, 0, 0), radius=0.003)

def show_reference(st, n=60, focal=False):
    # Coarse n×n preview for the viewer: the full 250×250 reference would be ~33 MB of
    # geometry per mesh. Computations still use the full st.reference.
    prev = pl.reference_preview(st, n)
    kv.register_surface_mesh("Reference", *prev["reference"], color=(0.8, 0.8, 0.8), opacity=0.4)
    kv.register_surface_mesh("Sphere centres (reference)", *prev["sphere_centers"], color=(0.6, 0.02, 0.8))
    if focal:
        kv.register_surface_mesh("Focal surface 1", *prev["focal_1"], color=(0.8, 0, 0))
        kv.register_surface_mesh("Focal surface 2", *prev["focal_2"], color=(0, 0.8, 0))

def show_cross(name, pts, d1, d2, colors):
    for suffix, (P, E), color in zip(("_D1", "_D2"), glyphs.cross_field(pts, d1, d2, CROSS_LEN), colors):
        kv.register_curve_network(name + suffix, P, E, color=color)

def show_cross_field(st):
    pts, t1, t2, lc1, lc2 = pl.cross_field(st)
    show_cross("L-conjugate tangents", pts, t1, t2, ((1, 0, 0), (0, 0, 1)))
    show_cross("L-conjugate rulings", pts, lc1, lc2, ((0, 0, 0), (1, 1, 1)))
    C, F = pl.sphere_centers(st)
    kv.register_surface_mesh("Sphere centres", C, F, color=(0, 1, 0), opacity=0.6)

def show_range(st, n_full=60):
    """Whole surface faint in the background, the selected (u, v) range coloured by K."""
    kv.clear()
    Vf, Ff = pl.surface_preview(st, n_full)
    kv.register_surface_mesh("Whole surface", Vf, Ff, color=(0.75, 0.75, 0.75), opacity=0.25)
    show_surface(st, clear=False)
    V, F, K, n = pl.surface_mesh(st)
    size = np.linalg.norm(V.max(axis=0) - V.min(axis=0))      # arrows at 8% of the range's size
    kv.add_vector_quantity("B-spline surface", "normals", n, defined_on="vertices", length=0.08 * size)


## Stage 0 — view the surface, choose the (u, v) range

Nothing is fitted yet. Move the sliders to choose the part of the surface and the number
of samples (the direction-field resolution); the viewer updates live — the whole surface
faint, the selected range coloured by Gaussian curvature, with its normals.
**swap u ↔ v** exchanges the roles of the two parameters, which flips the normals.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

st = pl.load_surface(SURFACE, EXPERIMENT, fit=False,
                     u_sample_num=EX["u_samples"], v_sample_num=EX["v_samples"],
                     interval_u=EX["interval_u"], interval_v=EX["interval_v"])
if EX["flip_normals"]:
    pl.flip_normals(st)                     # reverses u (as the original "Flip Normals")

w_u  = widgets.FloatRangeSlider(value=st.interval_u, min=0, max=1, step=0.01, description="u range", continuous_update=False)
w_v  = widgets.FloatRangeSlider(value=st.interval_v, min=0, max=1, step=0.01, description="v range", continuous_update=False)
w_nu = widgets.IntSlider(value=st.u_sample_num, min=5, max=200, description="u samples", continuous_update=False)
w_nv = widgets.IntSlider(value=st.v_sample_num, min=5, max=200, description="v samples", continuous_update=False)
w_swap = widgets.Checkbox(value=st.uvSwapped, description="swap u ↔ v (flip normals)")
w_info = widgets.Label()
_syncing = False

def _redraw():
    w_info.value = (f"{st.name}: u {st.interval_u[0]:.2f}–{st.interval_u[1]:.2f}, "
                    f"v {st.interval_v[0]:.2f}–{st.interval_v[1]:.2f}, "
                    f"{st.u_sample_num}x{st.v_sample_num} samples, "
                    f"normals {'flipped (u, v swapped)' if st.uvSwapped else 'as loaded'}")
    show_range(st)

def _on_range(change):
    if _syncing:
        return
    pl.set_sampling(st, interval_u=w_u.value, interval_v=w_v.value,
                    u_sample_num=w_nu.value, v_sample_num=w_nv.value)
    _redraw()

def _on_swap(change):
    global _syncing
    if bool(change["new"]) != st.uvSwapped:
        pl.swap_uv(st)                      # swaps the ranges and sample counts too
        _syncing = True                     # move the sliders without re-triggering
        w_u.value, w_v.value = st.interval_u, st.interval_v
        w_nu.value, w_nv.value = st.u_sample_num, st.v_sample_num
        _syncing = False
    _redraw()

for w in (w_u, w_v, w_nu, w_nv):
    w.observe(_on_range, names="value")
w_swap.observe(_on_swap, names="value")

display(widgets.VBox([w_u, w_v, w_nu, w_nv, w_swap, w_info]))
_redraw()
kv.show()

## Stage 0b — fit the spheres

Fits the radius function (sphere radius r(u, v)) and the dense reference for the range
chosen above. Re-run this cell after changing the range, the samples or the u ↔ v swap.

In [ ]:
pl.set_radius_function(
    st,
    kind="safe",          # "safe" | "mid" | "offset"
    percentage=EX["radius_percentage"],   # safe: r = p / max(|k1|, |k2|)
    offset=-0.2,          # offset: r = constant
)
show_range(st)
show_reference(st)
C, F = pl.radius_centers(st)
kv.register_surface_mesh("Sphere centres (radius type)", C, F, color=(0, 0, 1), opacity=0.6)

## Stage 1 — optional B-spline optimisation

Optimises the B-spline so that the Gaussian curvature stays positive
(`bsplineOptimization_jax.optimize`, `epsilon_K = 0.5`, `epsilon_H = 0.001`), then refits the
spheres. Off by default.

In [ ]:
RUN_BSPLINE_OPT = False
if RUN_BSPLINE_OPT:
    pl.optimize_bspline(st, w_curv=0.5, w_mean=0.001, w_lap=1.0, max_iter=40)
    pl.set_reference(st)      # refit the spheres for the new surface
    show_range(st)

EXPORT_NAME = None            # e.g. f"{SURFACE}_opt" to save data/bsplines/<name>.json
if EXPORT_NAME:
    print("saved", pl.export_bspline(st, EXPORT_NAME))
# pl.reset_bspline(st)        # back to the loaded surface

## Stage 2 — L-conjugate directions

`init_directions` sets the first-direction angle θ (method: `constant`, `linear`, `linearU`,
`linearV`, `cosineU`, `cosineV`, `boundary`) and computes its L-conjugate partner φ at every
sample. These directions go straight to the remesher in stage 3.

In [ ]:
st.initial_theta = EX["theta"]    # degrees, for "constant" (and "boundary")

pl.init_directions(st, method="constant")
show_surface(st)
show_cross_field(st)

In [ ]:
# Direction-field optimisation — NOT USED for the project (slow; the radius write-back of
# the original is broken, see pipeline.update_variables_from_optimizer). Kept for later.
#
# st.target_angle = 30.0      # degrees, "angle" term
# st.weight_DF.update({"lconjugacy": 1.1, "smooth": 1e-4, "angle": 0.0, "angleBoundary": 0.5})
# pl.setup_df_optimizer(st, legacy_regularizer=True)
# pl.run_df_optimizer(st, max_its=40,
#                     callback=lambda s, it: show_cross_field(s) if (it + 1) % 10 == 0 else None)
# show_cross_field(st)

In [ ]:
# Optional: cross-field singularities (index ±1/4)
V, sing_v, sing_idx = pl.compute_singularities(st)
if len(sing_v):
    kv.register_point_cloud("Singularities (+)", V[sing_v[sing_idx > 0]], color=(1, 0.2, 0.2), radius=0.02)
    kv.register_point_cloud("Singularities (-)", V[sing_v[sing_idx < 0]], color=(0.2, 0.2, 1), radius=0.02)
print(f"{len(sing_v)} singularities")

## Stage 3 — remeshing (external quad remesher)

`save_cross_field` writes `mesh.obj` and the two direction fields (`t` = curve tangents,
`lc` = rulings). `run_remesher` calls the external binary (`st.path_remesher`) for both,
in "integration" or "direct" mode, and saves `pre_remesh_state.pkl`. The remesher can also
be run by hand on the written files; `load_remeshed_surfaces` then reads the results.

**The remesher is interactive:** for each field (first `t`, then `lc`) it opens a viewer
window — press **S** in it to save the quad mesh, then close the window. The cell waits
until both windows are closed (the window may open behind VS Code).


In [ ]:
# st.path_remesher = "/path/to/quadRemesher"
pl.save_cross_field(st)
ok = pl.run_remesher(st, integrated=True)
print("remeshed:", ok, "->", st.save_path)

In [ ]:
if pl.load_remeshed_surfaces(st):
    kv.clear()
    show_surface(st, clear=False)
    kv.register_surface_mesh("Quads: L-conjugate tangents", st.quadRemeshed_t.vertices,
                             st.quadRemeshed_t.faces, color=(0.8, 0.8, 0.0), show_edges=True)
    kv.register_surface_mesh("Quads: rulings", st.quadRemeshed_lc.vertices,
                             st.quadRemeshed_lc.faces, color=(0.0, 0.8, 0.8), show_edges=True)
    kv.set_enabled("Quads: rulings", False)
else:
    print("no t_Remeshed.obj / lc_Remeshed.obj in", st.save_path)

## Stage 4 — L-mesh on the remeshed quads

`initialize_lmesh` builds the L-mesh on the `t` quads, as in the original app: one sphere
per quad face (centre on the reference sphere-centre surface at the quad barycentre, radius =
distance to the surface), one oriented plane per **inner** quad vertex (tangent plane of the
reference). In the swapped `Lmesh` this is `combinatorics="dual"`; the original also had
planes on the boundary vertices, which the dual has no place for.

If the kernel was restarted after stage 3, run the setup cells and stage 0, then uncomment
the `load_pre_remesh_state` line below to restore stages 0b–3 from `pre_remesh_state.pkl`.

In [ ]:
# pl.load_pre_remesh_state(st); pl.load_remeshed_surfaces(st)   # after a kernel restart

lm = pl.initialize_lmesh(st)
print(f"{len(lm.top.vertices)} spheres, {len(lm.PF)} planes, "
      f"oriented contact deviation {lm.compute_oriented_contact_dev():.3e}")

def show_lmesh_centers(st, name="Sphere centres (L-mesh)"):
    C, F = pl.lmesh_centers(st)
    kv.register_surface_mesh(name, C, F, color=(0.8, 0.8, 0.8), show_edges=True)
    kv.register_point_cloud("Centres", C, color=(0.8, 0.0, 0.1), radius=0.004)

kv.clear()
kv.register_surface_mesh("Quads: L-conjugate tangents", st.quadRemeshed_t.vertices,
                         st.quadRemeshed_t.faces, color=(0.8, 0.8, 0.0), show_edges=True)
show_lmesh_centers(st)

SHOW_FULL_LMESH = True
if SHOW_FULL_LMESH:
    from LMesh.lmesh_serializers import lmesh_scene
    kv.push_objects(lmesh_scene(st.lmesh))

### Post-optimisation

The original panel: oriented contact, plane / sphere fairness (`quadFair`, `quadFairSph`),
tangential distance, contact points close to and gliding on the reference, a
previous-iteration regulariser and unit plane normals. "Last Opt" (`finalize_lmesh`) then
runs 20 steps with oriented contact only. `legacy_regularizer=True` keeps the original
regulariser (r = X_k − X_{k−1}).
As in the paper, both fairness weights are multiplied by 0.1 every ten steps (full weight
for steps 0–9); each run of the cell starts again from the full weights.

In [ ]:
pl.setup_lmesh_optimizer(st, legacy_regularizer=True)

In [ ]:
st.weight_lmesh.update({
    "or_contact": 1.0, "tangentialDist": 0, "quadFair": 1e-3, "quadFairSph": 1e-3,
    "contactProximity": 0, "contactGliding": 0, "cv_proximity": 0.0, "cv_gliding": 0.0,
})
pl.run_lmesh_optimizer(st, max_its=100,
                       callback=lambda s, it: show_lmesh_centers(s) if (it + 1) % 10 == 0 else None)
print(f"oriented contact deviation {st.lmesh.compute_oriented_contact_dev():.3e}")
show_lmesh_centers(st)

In [ ]:
pl.finalize_lmesh(st, its=20)          # "Last Opt": oriented contact only
print(f"oriented contact deviation {st.lmesh.compute_oriented_contact_dev():.3e}")
from LMesh.lmesh_serializers import lmesh_scene
kv.push_objects(lmesh_scene(st.lmesh))

In [ ]:
print("exported ->", pl.export_lmesh(st, "lmesh"))